Start

In [1]:
import cv2
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import os

Thresholding

In [48]:
img = cv2.imread('images/queimada/queimada4.jpg')
# img = cv2.imread('images/natural/natural3.jpg')

_, thresh_img = cv2.threshold(img, 150, 255, cv2.THRESH_BINARY)

cv2.imshow('Imagem Segmentada', thresh_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [47]:
img = cv2.imread('images/queimada/queimada4.jpg')
# img = cv2.imread('images/natural/natural3.jpg')

hsv_img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

_, thresh_img = cv2.threshold(hsv_img[:, :, 0], 10, 255, cv2.THRESH_BINARY)

cv2.imshow('Imagem Segmentada', thresh_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
def fireDetect(img):
    # Conversão para escala de cinza
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Filtros
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    gray = cv2.medianBlur(gray, 5)
    
    # Histograma e thresholding
    hist, bins = np.histogram(gray, 256, [0, 256])
    threshold = 0.5 * (bins[1] + bins[-1])
    gray[gray < threshold] = 0
    gray[gray >= threshold] = 255
    
    # Segmentação por cor
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lower_red = np.array([0, 100, 100])
    upper_red = np.array([10, 255, 255])
    mask = cv2.inRange(hsv, lower_red, upper_red)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        area = cv2.contourArea(contour)
        if area > 1000:
            x, y, w, h = cv2.boundingRect(contour)
            cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
    
    # Análise de características
    features = []
    for contour in contours:
        area = cv2.contourArea(contour)
        perimeter = cv2.arcLength(contour, True)
        features.append([area, perimeter])
    
    try:
        # Classificação
        scaler = StandardScaler()
        features = scaler.fit_transform(features)
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(features, np.ones(len(features)))
        
        
        # Predição
        prediction = rf.predict(features)
        print(prediction)
                
        # Post-processamento
        for i, pred in enumerate(prediction):
            if pred == 1:
                x, y, w, h = cv2.boundingRect(contours[i])
                cv2.rectangle(img, (x, y), (x + w, y + h), (0, 0, 255), 2)
        
        
        # Mostrar imagem
        print("FOGO!")
        cv2.imshow('Fogo', img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    except:
        print("Não fogo")
        
        
for img in os.listdir("images/test/images"):
    fireDetect(cv2.imread(f"images/test/images/{img}"))
#     break

fireDetect(cv2.imread("images/queimada/queimada4.jpg"))

[1. 1. 1.]
FOGO!


In [18]:
def shannon_entropy(data):
    data = data.flatten()
    data = data[data > 0]
    data = data / data.sum()
    entropy = -np.sum(data * np.log2(data))
    return entropy

def fireDetect(img):
    height, width, _ = img.shape
    
    # Conversão para escala de cinza
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Filtros
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    gray = cv2.medianBlur(gray, 5)
    
    # Histograma e thresholding
    hist, bins = np.histogram(gray, 256, [0, 256])
    threshold = 0.5 * (bins[1] + bins[-1])
    gray[gray < threshold] = 0
    gray[gray >= threshold] = 255
    
    # Segmentação por contorno
    contours, _ = cv2.findContours(gray, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    features = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        if x >= 0 and y >= 0 and x + w <= width and y + h <= height:
            area = cv2.contourArea(contour)
            perimeter = cv2.arcLength(contour, True)
            if perimeter > 0:
                texture = np.mean(gray[y:y+h, x:x+w])
                shape = area / perimeter
                features.append([area, perimeter, texture, shape])
    
    # Regras de entropia
    threshold_entropy = 0.5
    for i, feature in enumerate(features):
        area, perimeter, texture, shape = feature
        entropy = shannon_entropy(gray[y:y+h, x:x+w])
        if entropy > threshold_entropy:
            x, y, w, h = cv2.boundingRect(contours[i])
            cv2.rectangle(img, (x, y), (x + w, y + h), (0, 0, 255), 2)


    
    # Classificação
    scaler = StandardScaler()
    features = scaler.fit_transform(features)
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(features, np.ones(len(features)))
    
    # Predição
    prediction = rf.predict(features)
    
    # Post-processamento
    # for i, pred in enumerate(prediction):
    #     if pred == 1:
    #         x, y, w, h = cv2.boundingRect(contours[i])
    #         cv2.rectangle(img, (x, y), (x+w, y+h), (0, 0, 255), 2)

    
    # Mostrar imagem
    cv2.imshow('Fogo', img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    

for img in os.listdir("images/test/images"):
    fireDetect(cv2.imread(f"images/test/images/{img}"))
    break

fireDetect(cv2.imread("images/queimada/queimada4.jpg"))